# 09 · DQ reconciliation (fail-fast gate)
Scores `silver.dq_issue_log` against `DQ_ANSWER_KEY.json` (module 08). Fails the task if recall < 100%, so Gold is skipped in the job.

✅ Expect **1,786 / 1,786** matched. Run interactively with `run_id = manual` to score the latest run.

In [ ]:
from pyspark.sql import functions as F
from hcm_utils import load_config, resolve_paths

dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("run_id", "manual")
catalog = dbutils.widgets.get("catalog")
run_id  = dbutils.widgets.get("run_id")
cfg = load_config()

bronze_rows = dbutils.jobs.taskValues.get(taskKey="bronze_ingest", key="bronze_total_rows", debugValue=0)
print("bronze rows this run:", bronze_rows)

In [ ]:
KEY = ["SourceFile", "IssueType", "RowRef"]

expected = (spark.read.option("multiLine", "true").json(resolve_paths(cfg, catalog)["dq_answer_key"])
            .select(F.col("file").alias("SourceFile"),
                    F.col("issue_type").alias("IssueType"),
                    F.col("row_ref").alias("RowRef"))
            .distinct())

log = spark.table(f"{catalog}.silver.dq_issue_log")
if run_id == "manual":
    run_id = log.orderBy(F.col("DetectedAt").desc()).first()["RunId"]
detected = log.filter(F.col("RunId") == run_id).select(*KEY).distinct()

matched = expected.join(detected, KEY, "inner")
missed  = expected.join(detected, KEY, "left_anti")
extra   = detected.join(expected, KEY, "left_anti")

def by_type(df, name):
    return df.groupBy("SourceFile", "IssueType").agg(F.count("*").alias(name))

report = (by_type(expected, "Expected")
          .join(by_type(detected, "Detected"), ["SourceFile", "IssueType"], "full")
          .join(by_type(matched, "Matched"), ["SourceFile", "IssueType"], "left")
          .fillna(0)
          .withColumn("RecallPct", F.round(100 * F.col("Matched") / F.when(F.col("Expected") > 0, F.col("Expected")), 1))
          .withColumn("PrecisionPct", F.round(100 * F.col("Matched") / F.when(F.col("Detected") > 0, F.col("Detected")), 1))
          .withColumn("RunId", F.lit(run_id))
          .withColumn("ReconciledAt", F.current_timestamp())
          .orderBy("SourceFile", "IssueType"))

display(report)
report.write.mode("append").saveAsTable(f"{catalog}.silver.dq_reconciliation")

In [ ]:
spark.sql(f"""
CREATE OR REPLACE VIEW {catalog}.gold.v_dq_scorecard AS
SELECT RunId, ReconciledAt, SourceFile, IssueType, Expected, Detected, Matched, RecallPct, PrecisionPct,
       CASE IssueType
         WHEN 'NULL_VALUE' THEN 'Completeness'
         WHEN 'DUPLICATE_ROW' THEN 'Uniqueness'
         WHEN 'DUPLICATE_ROW_CHANGED' THEN 'Uniqueness'
         WHEN 'MULTI_ASSIGNMENT' THEN 'Uniqueness'
         WHEN 'CONSISTENCY_VIOLATION' THEN 'Consistency'
         WHEN 'STALE_TIMESTAMP' THEN 'Timeliness'
         ELSE 'Validity' END AS DqDimension
FROM {catalog}.silver.dq_reconciliation""")

In [ ]:
n_exp, n_match = expected.count(), matched.count()
print(f"Expected {n_exp:,} | Detected {detected.count():,} | Matched {n_match:,} | "
      f"Missed {missed.count():,} | Extra {extra.count():,}")
if missed.count():
    display(missed.limit(50))
dbutils.jobs.taskValues.set(key="dq_recall_pct", value=round(100 * n_match / n_exp, 2))
assert n_match == n_exp, f"DQ recall below 100%: {n_exp - n_match} known defects missed"